# 1. INE data for Andalusia

In [1]:
import logging
import pandas as pd
import sys
from pathlib import Path
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "src" / "ingesta").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project root containing src/ingesta.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.common.storage import read_csv_dataset
from src.ingesta.pipeline import (
    download_and_upload_gyms,
    download_and_upload_income,
    download_and_upload_municipality_boundaries,
    download_and_upload_population,
)

def preview_table(csv_path, columns=None, rows=5):
    if csv_path is None:
        print("No dataset was persisted; check the SAVE_LOCALLY and SAVE_S3 settings.")
        return
    preview = read_csv_dataset(csv_path, nrows=rows, dtype=str, keep_default_na=False)
    if columns is not None:
        preview_columns = [column for column in columns if column in preview.columns]
        if not preview_columns:
            raise ValueError(f"None of the requested preview columns exist in {csv_path}.")
        preview = preview.loc[:, preview_columns]
    display(preview)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

## Andalusian population, all ages and sexes, latest five years

In [2]:
population_csv = download_and_upload_population(number_of_years=5)

2026-10-20 17:28:06,312 INFO src.ingesta.datos_poblacion.ine.common: Downloaded INE table 68542.
2026-10-20 17:29:04,075 INFO src.ingesta.datos_poblacion.ine.padron: Saved 1201050 Andalusian population rows for years [2025, 2024, 2023, 2022, 2021] to C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\raw\population_andalusia.csv.


In [3]:
preview_table(population_csv)

,Provincias,Municipios,Sexo,Edad,Periodo,Total,source,downloaded_at_utc
0,04 Almería,04001 Abla,Hombres,Todas las edades,2025,660,INE,2026-10-20T15:28:06.312532+00:00
1,04 Almería,04001 Abla,Hombres,Todas las edades,2024,658,INE,2026-10-20T15:28:06.312532+00:00
2,04 Almería,04001 Abla,Hombres,Todas las edades,2023,652,INE,2026-10-20T15:28:06.312532+00:00
3,04 Almería,04001 Abla,Hombres,Todas las edades,2022,643,INE,2026-10-20T15:28:06.312532+00:00
4,04 Almería,04001 Abla,Hombres,Todas las edades,2021,648,INE,2026-10-20T15:28:06.312532+00:00


## Municipal mean net income, latest year

In [4]:
income_csv = download_and_upload_income(number_of_years=1)

2026-10-20 17:30:26,724 INFO src.ingesta.datos_poblacion.ine.common: Downloaded INE table 30824.
2026-10-20 17:30:49,060 INFO src.ingesta.datos_poblacion.ine.renta: Saved 786 Andalusian income rows for years [2024] to C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\raw\income_andalusia.csv.


In [5]:
preview_table(income_csv)

,Municipios,Distritos,Secciones,Indicadores de renta media,Periodo,Total,source,downloaded_at_utc
0,04001 Abla,,,Renta neta media por persona,2024,12.836,INE,2026-10-20T15:30:26.724517+00:00
1,04002 Abrucena,,,Renta neta media por persona,2024,12.565,INE,2026-10-20T15:30:26.724517+00:00
2,04003 Adra,,,Renta neta media por persona,2024,10.965,INE,2026-10-20T15:30:26.724517+00:00
3,04004 Albanchez,,,Renta neta media por persona,2024,12.885,INE,2026-10-20T15:30:26.724517+00:00
4,04005 Alboloduy,,,Renta neta media por persona,2024,11.848,INE,2026-10-20T15:30:26.724517+00:00


# 2. OpenStreetMap gyms

In [6]:
from src.ingesta.pipeline import download_and_upload_gyms

In [7]:
gyms_csv = download_and_upload_gyms()

2026-10-20 17:30:49,219 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Almería.
2026-10-20 17:31:38,381 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Cádiz.
2026-10-20 17:32:23,932 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Córdoba.
2026-10-20 17:33:09,146 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Granada.
2026-10-20 17:33:56,828 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Huelva.
2026-10-20 17:34:42,046 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Jaén.
2026-10-20 17:34:42,410 WARNING src.ingesta.common.ingestion: OpenStreetMap request to https://overpass-api.de/api/interpreter failed (attempt 1/3): 502 Server Error: Bad Gateway for url: https://overpass-api.de/api/interpreter
2026-10-20 17:34:42,412 INFO src.ingesta.common.ingestion: Retrying in 30 seconds.
2026-10-20 17:35:12,603 WARNING src.ingesta.common.ingestion: OpenStreetMap request to https://overpass-api.de/api/interpreter failed (attempt 2/3): 502 Se

In [8]:
preview_table(
    gyms_csv,
    columns=[
        "osm_type", "osm_id", "province", "latitude", "longitude",
        "tag_name", "tag_leisure", "tag_sport", "source", "downloaded_at_utc",
    ],
)

,osm_type,osm_id,province,latitude,longitude,tag_name,tag_leisure,tag_sport,source,downloaded_at_utc
0,node,260812441,Almería,37.1720276,-1.8339941,Go-Karting,sports_centre,karting,OpenStreetMap,2026-10-20T15:31:28.373903+00:00
1,node,1620487480,Almería,37.3453627,-2.4325406,Pabellón,sports_centre,,OpenStreetMap,2026-10-20T15:31:28.373903+00:00
2,node,1912986759,Almería,37.212201,-1.8108268,Vera Costa,fitness_station,fitness,OpenStreetMap,2026-10-20T15:31:28.373903+00:00
3,node,2452253961,Almería,36.8380866,-2.9492787,Pabellon de deportes,sports_centre,,OpenStreetMap,2026-10-20T15:31:28.373903+00:00
4,node,4432638479,Almería,36.9699743,-2.6075333,Frontón de Huécija,sports_centre,pelota,OpenStreetMap,2026-10-20T15:31:28.373903+00:00


# 3. CNIG municipality boundaries

In [9]:
from src.ingesta.pipeline import download_and_upload_municipality_boundaries

In [10]:
municipality_boundaries_csv = download_and_upload_municipality_boundaries()

2026-10-20 17:40:35,554 INFO src.ingesta.limites_municipios.cnig: Downloading CNIG page at index 0.
2026-10-20 17:41:47,413 INFO src.ingesta.limites_municipios.cnig: Downloading CNIG page at index 500.
2026-10-20 17:42:23,747 INFO src.ingesta.limites_municipios.cnig: Downloading CNIG page at index 786.
2026-10-20 17:42:24,303 INFO src.ingesta.limites_municipios.cnig: Saved 786 Andalusian municipality boundaries to C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\raw\municipality_boundaries_andalusia.csv.


In [11]:
preview_table(
    municipality_boundaries_csv,
    columns=["nationalCode", "name", "geometry", "source", "downloaded_at_utc"],
)

,nationalCode,name,geometry,source,downloaded_at_utc
0,34010404001,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-2...",IGN/CNIG BDDAE,2026-10-20T15:41:43.805343+00:00
1,34010404002,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-2...",IGN/CNIG BDDAE,2026-10-20T15:41:43.805343+00:00
2,34010404003,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-3...",IGN/CNIG BDDAE,2026-10-20T15:41:43.805343+00:00
3,34010404004,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-2...",IGN/CNIG BDDAE,2026-10-20T15:41:43.805343+00:00
4,34010404005,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-2...",IGN/CNIG BDDAE,2026-10-20T15:41:43.805343+00:00
